# 价格合适

### 现在，评估我们微调的开源模型

In [ ]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
# 进口

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

In [ ]:
# 常数

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "denis-mutuma" # your HF name here!

LITE_MODE = True

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2026-03-08_13.42.08-lite"
  REVISION = None
# 别的：
# RUN_NAME =“2025-11-28_18.47.07”
# 修订版=“b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd”

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# 超参数 - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

### 登录 HuggingFace

如果您还没有 HuggingFace 帐户，请访问 https://huggingface.co 注册并创建令牌。

然后通过单击左侧的钥匙图标选择此笔记本的 Secrets，并添加一个名为“HF_TOKEN”的新密钥，并将该值作为您的令牌。

In [ ]:
# 登录 HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [ ]:
test[0]

## 现在加载分词器和模型

In [ ]:
# 选择正确的量化

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# 加载分词器和模型

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 使用 PEFT 加载微调模型
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
fine_tuned_model

# 关键时刻！

## 在推理模式下使用模型

试图以 87.62 美元的价格超越“人类”的表现水平

或者可能接近 gpt-4.1-nano，价格为 62.51 美元

## 警告

请记住，商品的价格差异很大；该模型无法预测它没有任何信息的事物，例如销售价格。

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test)

In [ ]:
import plotly.graph_objects as go

results = [
    ("NLP + LR", "gray", 75.25),
    ("Random Forest", "gray", 72.13),
    ("XGBoost", "gray", 65.32),
    ("Neural Network", "orange", 71.56),
    ("GPT 4.1 Nano", "slateblue", 86.54),
    ("GPT 5.1", "green", 41.01),
    ("GPT 4.1 Nano (Fine-tuned)", "red", 84.82),
    ("Llama-3.2-3B (QLoRA)", "blue", 61.79)
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Week 6 exercise - Denis Mutuma",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()